# BT4012 — first temporal baseline

Predict illicit Bitcoin transactions using **ROC-AUC** as the competition metric.
This notebook uses only labelled observations and `feat_1`–`feat_165`. No graph
features, GNN, pseudo-labels, random primary split, or hyperparameter search.

The user confirms **0 = negative, 1 = positive / illicit**. The raw CSVs do not
encode semantic class names. Restart the kernel and run every cell to reproduce
validation and write a Kaggle-ready CSV. Raw files are read-only; nothing is submitted.

## 1. Imports and configuration

Use a fixed seed and one numerical thread. Paths work when Jupyter starts from the
repository root or `notebooks/`. Package versions and raw hashes are saved with
aggregate results in `outputs/baseline_metrics.json`.

In [1]:
from pathlib import Path
import hashlib
import json
import platform
import warnings
from importlib.metadata import version

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.base import clone
from sklearn.exceptions import ConvergenceWarning
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from threadpoolctl import threadpool_limits

SEED = 42
FEATURES = [f"feat_{i}" for i in range(1, 166)]
THRESHOLD = 0.5
np.random.seed(SEED)

ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "data" / "raw" / "train.csv").is_file()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Start Jupyter inside the project and place CSVs in data/raw/.")
RAW_DIR = ROOT / "data" / "raw"
OUTPUT_DIR = ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)
SUBMISSION_PATH = OUTPUT_DIR / "submission_baseline.csv"
VERSIONS = {name: version(name) for name in ["numpy", "pandas", "scikit-learn", "threadpoolctl"]}
print("Python:", platform.python_version())
print("Packages:", VERSIONS)
print("Features:", len(FEATURES), "| seed:", SEED)

Python: 3.12.4
Packages: {'numpy': '1.26.4', 'pandas': '2.2.2', 'scikit-learn': '1.5.1', 'threadpoolctl': '3.5.0'}
Features: 165 | seed: 42


## 2. Load data

Load complete CSVs. The edge list is inspected for its schema and size only; it is
never joined to model features. Hashes identify the exact inputs and detect changes
relative to the reviewed initial manifest.

In [2]:
train = pd.read_csv(RAW_DIR / "train.csv", low_memory=False)
test = pd.read_csv(RAW_DIR / "test.csv", low_memory=False)
sample = pd.read_csv(RAW_DIR / "sample_submission.csv")
edges = pd.read_csv(RAW_DIR / "txs_edgelist.csv")

raw_hashes = {}
for name in ["train.csv", "test.csv", "txs_edgelist.csv", "sample_submission.csv"]:
    with (RAW_DIR / name).open("rb") as stream:
        raw_hashes[name] = hashlib.file_digest(stream, "sha256").hexdigest()
manifest = json.loads((ROOT / "docs" / "raw_manifest.json").read_text())
changed_inputs = [name for name, digest in raw_hashes.items()
                  if digest != manifest["files"][name]["sha256"]]
if changed_inputs:
    print("NOTICE: raw files differ from the initial manifest:", changed_inputs)
    print("All checks and metrics below use the current files; review changes before comparing runs.")

display(pd.DataFrame([
    {"file": name, "rows": len(frame), "columns": len(frame.columns)}
    for name, frame in [("train.csv", train), ("test.csv", test),
                        ("txs_edgelist.csv", edges), ("sample_submission.csv", sample)]
]))

,file,rows,columns
0,train.csv,141772,168
1,test.csv,15329,168
2,txs_edgelist.csv,234355,2
3,sample_submission.csv,15329,2


## 3. Dataset sanity checks

Verify schemas, row counts, labels, time coverage, feature quality, and keys from
actual files. Expected counts are comparisons, not substitutes for inspecting data.
Structural problems stop execution to avoid silently producing incorrect predictions.
The graph direction `txId1 -> txId2` follows the user-provided convention.

In [3]:
assert train.columns.tolist() == ["txId", "time_step", *FEATURES, "label"]
assert test.columns.tolist() == ["index", "txId", "time_step", *FEATURES]
assert edges.columns.tolist() == ["txId1", "txId2"]
assert sample.columns.tolist() == ["index", "target"]
assert set(train["label"].dropna().unique()) == {0, 1}
assert train["label"].isna().any() and train["label"].notna().any()

for frame in [train, test]:
    assert frame["txId"].notna().all() and frame["txId"].is_unique
    assert frame["time_step"].notna().all()
    assert pd.api.types.is_integer_dtype(frame["time_step"])
    assert all(pd.api.types.is_numeric_dtype(frame[c]) for c in FEATURES)
assert set(train["txId"]).isdisjoint(test["txId"])
for frame in [test, sample]:
    assert frame["index"].notna().all() and frame["index"].is_unique
assert len(test) == len(sample)
assert set(test["index"]) == set(sample["index"])
assert train["time_step"].between(1, 35).all()
assert test["time_step"].min() > train["time_step"].max()

observations = [
    ("train rows", 141772, len(train)),
    ("test rows", 15329, len(test)),
    ("features", 165, len([c for c in train if c.startswith("feat_")])),
    ("label 0", 27591, int(train["label"].eq(0).sum())),
    ("label 1", 3644, int(train["label"].eq(1).sum())),
    ("train time steps", list(range(1, 36)), sorted(train["time_step"].unique().tolist())),
    ("test time steps", list(range(36, 50)), sorted(test["time_step"].unique().tolist())),
]
sanity = pd.DataFrame(observations, columns=["check", "expected", "actual"])
sanity["matches"] = [expected == actual for _, expected, actual in observations]
display(sanity)
if not sanity["matches"].all():
    print("DISCREPANCY: see comparisons above. Subsequent results use actual observed data.")

quality = []
for name, frame in [("train", train), ("test", test)]:
    quality.append({"dataset": name, "missing_feature_values": int(frame[FEATURES].isna().sum().sum()),
                    "infinite_feature_values": int(np.isinf(frame[FEATURES].to_numpy()).sum()),
                    "duplicate_complete_rows": int(frame.duplicated().sum())})
display(pd.DataFrame(quality))
print("Edge rows inspected:", len(edges), "| schema: txId1 -> txId2; no graph features used")
del edges

,check,expected,actual,matches
0,train rows,141772,141772,True
1,test rows,15329,15329,True
2,features,165,165,True
3,label 0,27591,27591,True
4,label 1,3644,3644,True
5,train time steps,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...",True
6,test time steps,"[36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 4...","[36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 4...",True


,dataset,missing_feature_values,infinite_feature_values,duplicate_complete_rows
0,train,0,0,0
1,test,0,0,0


Edge rows inspected: 234355 | schema: txId1 -> txId2; no graph features used


## 4. Label analysis

Only non-null training labels are eligible for supervised learning. Unknown labels
are excluded, never filled with zero. The 0.5 values in the sample submission are
placeholders, not training targets.

In [4]:
labelled = train.loc[train["label"].notna()].copy()
labelled["label"] = labelled["label"].astype("int64")
label_counts = labelled["label"].value_counts().reindex([0, 1], fill_value=0)
label_summary = pd.DataFrame({"rows": label_counts, "percent": 100 * label_counts / len(labelled)})
label_summary.index.name = "label"
display(label_summary)
print("Labelled:", len(labelled), "| excluded unlabelled:", int(train["label"].isna().sum()))

,rows,percent
label,,
0,27591,88.3336
1,3644,11.6664


Labelled: 31235 | excluded unlabelled: 110537


## 5. Temporal split

Fit using labelled time steps **1–30**, then validate on labelled time steps
**31–35**. This tests prediction into a later period, matching the competition's
future-period setup. Both classes must be present. This split is fixed before fitting.

In [5]:
fit_rows = labelled.loc[labelled["time_step"] <= 30].copy()
validation_rows = labelled.loc[labelled["time_step"].between(31, 35)].copy()
assert len(fit_rows) + len(validation_rows) == len(labelled)
assert fit_rows["time_step"].max() < validation_rows["time_step"].min()
assert set(fit_rows["txId"]).isdisjoint(validation_rows["txId"])
assert fit_rows["label"].nunique() == validation_rows["label"].nunique() == 2

split_records = []
for name, frame in [("fit: time <= 30", fit_rows), ("validation: time 31–35", validation_rows)]:
    split_records.append({"split": name, "rows": len(frame),
                          "class_0": int(frame["label"].eq(0).sum()),
                          "class_1": int(frame["label"].eq(1).sum()),
                          "positive_percent": float(frame["label"].mean() * 100)})
split_summary = pd.DataFrame(split_records)
display(split_summary)

,split,rows,class_0,class_1,positive_percent
0,fit: time <= 30,26905,23951,2954,10.979372
1,validation: time 31–35,4330,3640,690,15.935335


## 6. Preprocessing

Select the 165 named features explicitly. IDs, `label`, and `time_step` are excluded.
Replace infinite values with missing values in working copies only. Median
imputation and standardization are fitted inside the pipeline using fitting rows
only. No missing/infinite feature values were observed in the initial files, but
these steps make handling explicit. A completely missing fitting column stops the
run because a meaningful fitting-period median cannot be estimated.

In [6]:
X_fit = fit_rows[FEATURES].replace([np.inf, -np.inf], np.nan)
y_fit = fit_rows["label"]
X_validation = validation_rows[FEATURES].replace([np.inf, -np.inf], np.nan)
y_validation = validation_rows["label"]
X_all = labelled[FEATURES].replace([np.inf, -np.inf], np.nan)
y_all = labelled["label"]
X_test = test[FEATURES].replace([np.inf, -np.inf], np.nan)

assert X_fit.columns.tolist() == X_validation.columns.tolist() == X_test.columns.tolist() == FEATURES
assert not {"txId", "index", "time_step", "label"}.intersection(FEATURES)
if X_fit.isna().all().any():
    raise ValueError("All-missing fitting columns: " + str(X_fit.columns[X_fit.isna().all()].tolist()))
print("Using", X_fit.shape[1], "features; all preprocessing statistics will be learned inside Pipeline.fit().")

Using 165 features; all preprocessing statistics will be learned inside Pipeline.fit().


## 7. Baseline model

Use L2-regularized logistic regression with `C=1`, the deterministic `lbfgs` solver,
`max_iter=3000`, and a fixed seed. `class_weight='balanced'` gives the two classes
 equal total training weight because illicit rows are the minority. This is a
predeclared baseline choice, not a claim that weighting maximizes ROC-AUC. Weighted
probabilities need not be calibrated to the observed illicit prevalence.

Convergence warnings are treated as errors rather than ignored. The numerical
thread count is limited to one for consistent execution.

In [7]:
pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        C=1.0, penalty="l2", solver="lbfgs", class_weight="balanced",
        max_iter=3000, tol=1e-4, random_state=SEED,
    )),
])
with warnings.catch_warnings(), threadpool_limits(limits=1):
    warnings.simplefilter("error", ConvergenceWarning)
    pipeline.fit(X_fit, y_fit)

assert pipeline.named_steps["scaler"].n_samples_seen_ == len(X_fit)
assert pipeline.named_steps["model"].classes_.tolist() == [0, 1]
fit_iterations = int(pipeline.named_steps["model"].n_iter_.max())
print("Validation model fitted successfully. Solver iterations:", fit_iterations)

Validation model fitted successfully. Solver iterations: 191


## 8. Validation results

**ROC-AUC** measures ranking using continuous class-1 probabilities and does not
use the 0.5 classification threshold. **Precision, recall, F1, and the confusion
matrix** below use threshold **0.5**, with class 1 as positive. No threshold tuning
is performed. The confusion matrix uses rows = actual and columns = predicted.

In [8]:
positive_column = int(np.flatnonzero(pipeline.named_steps["model"].classes_ == 1)[0])
with threadpool_limits(limits=1):
    validation_probabilities = pipeline.predict_proba(X_validation)[:, positive_column]
assert np.isfinite(validation_probabilities).all()
validation_predictions = (validation_probabilities >= THRESHOLD).astype(int)
validation_metrics = {
    "roc_auc": float(roc_auc_score(y_validation, validation_probabilities)),
    "precision_at_0_5": float(precision_score(y_validation, validation_predictions, zero_division=0)),
    "recall_at_0_5": float(recall_score(y_validation, validation_predictions, zero_division=0)),
    "f1_at_0_5": float(f1_score(y_validation, validation_predictions, zero_division=0)),
}
cm = confusion_matrix(y_validation, validation_predictions, labels=[0, 1])
display(pd.Series(validation_metrics, name="validation value").to_frame())
display(pd.DataFrame(cm, index=["actual 0", "actual 1"], columns=["predicted 0", "predicted 1"]))
print("Constant-score reference ROC-AUC:", roc_auc_score(y_validation, np.full(len(y_validation), 0.5)))
print("Primary validation ROC-AUC:", f"{validation_metrics['roc_auc']:.6f}")

,validation value
roc_auc,0.917141
precision_at_0_5,0.342944
recall_at_0_5,0.982609
f1_at_0_5,0.508436


,predicted 0,predicted 1
actual 0,2341,1299
actual 1,12,678


Constant-score reference ROC-AUC: 0.5
Primary validation ROC-AUC: 0.917141


## 9. Retrain using all labelled training data

Clone the same pipeline configuration and refit every preprocessing/model step on
**all labelled time steps 1–35**. The validation metrics above belong to the earlier
fit; the final model is for test prediction and has no measured test ROC-AUC.
Balanced weights are recomputed from the final fitting labels, as specified by
`class_weight='balanced'`. No unlabelled or test observations are used for fitting.

In [9]:
assert labelled["time_step"].between(1, 35).all()
final_pipeline = clone(pipeline)
with warnings.catch_warnings(), threadpool_limits(limits=1):
    warnings.simplefilter("error", ConvergenceWarning)
    final_pipeline.fit(X_all, y_all)
assert final_pipeline.named_steps["scaler"].n_samples_seen_ == len(labelled)
assert final_pipeline.named_steps["model"].classes_.tolist() == [0, 1]
final_iterations = int(final_pipeline.named_steps["model"].n_iter_.max())
print("Final model fitted on", len(labelled), "labelled rows. Solver iterations:", final_iterations)

Final model fitted on 31235 labelled rows. Solver iterations: 209


## 10. Generate Kaggle submission

Predict the probability of **class 1 (illicit)**. Join probabilities to the template
using the CSV `index` column so that row ordering is explicitly preserved even if
test rows are reordered. Keep exactly `index,target`; validate before writing.

In [10]:
positive_column = int(np.flatnonzero(final_pipeline.named_steps["model"].classes_ == 1)[0])
with threadpool_limits(limits=1):
    test_probabilities = final_pipeline.predict_proba(X_test)[:, positive_column]
predictions = pd.DataFrame({"index": test["index"].to_numpy(), "target": test_probabilities})
submission = sample[["index"]].merge(predictions, on="index", how="left", sort=False, validate="one_to_one")

assert submission.columns.tolist() == ["index", "target"]
assert len(submission) == len(test) == len(sample)
assert submission["index"].equals(sample["index"])
assert submission["index"].is_unique
assert submission["index"].notna().all()
assert submission["target"].notna().all()
assert np.isfinite(submission["target"]).all()
assert submission["target"].between(0, 1).all()
submission.to_csv(SUBMISSION_PATH, index=False)
print("Saved outputs/submission_baseline.csv with", len(submission), "rows and columns index,target.")

Saved outputs/submission_baseline.csv with 15329 rows and columns index,target.


## 11. Final sanity checks

Read back the saved CSV, repeat format/probability checks, and verify raw files
were unchanged. Save only aggregate metrics/configuration in the JSON run record.
The notebook displays no raw transaction rows or individual predictions. Output
files are ignored by Git. No Kaggle submission or network action is performed.

In [11]:
saved_submission = pd.read_csv(SUBMISSION_PATH)
assert saved_submission.columns.tolist() == ["index", "target"]
assert len(saved_submission) == len(test)
assert saved_submission["index"].equals(sample["index"])
assert saved_submission["index"].is_unique
assert not saved_submission.isna().any().any()
assert np.isfinite(saved_submission["target"]).all()
assert saved_submission["target"].between(0, 1).all()
np.testing.assert_allclose(saved_submission["target"], submission["target"], rtol=1e-12, atol=1e-15)
for name, original_hash in raw_hashes.items():
    with (RAW_DIR / name).open("rb") as stream:
        assert hashlib.file_digest(stream, "sha256").hexdigest() == original_hash, name

run_record = {
    "python": platform.python_version(), "packages": VERSIONS,
    "seed": SEED, "numerical_threads": 1, "features": FEATURES,
    "excluded_features": ["txId", "index", "label", "time_step"],
    "label_mapping": {"0": "negative", "1": "positive / illicit (user confirmed)"},
    "raw_sha256": raw_hashes, "raw_changed_from_initial_manifest": changed_inputs,
    "labelled_rows": len(labelled), "unlabelled_rows_excluded": int(train["label"].isna().sum()),
    "split_counts": split_records,
    "fit_time_steps": [1, 30], "validation_time_steps": [31, 35], "final_fit_time_steps": [1, 35],
    "preprocessing": {"infinity": "replace with NaN", "imputation": "training median", "scaling": "StandardScaler"},
    "model": final_pipeline.named_steps["model"].get_params(),
    "fit_iterations": fit_iterations, "final_fit_iterations": final_iterations,
    "validation": validation_metrics, "threshold": THRESHOLD,
    "confusion_matrix_actual_rows_predicted_columns": cm.tolist(),
    "submission": {"path": "outputs/submission_baseline.csv", "rows": len(saved_submission),
                   "min_probability": float(saved_submission["target"].min()),
                   "max_probability": float(saved_submission["target"].max()),
                   "mean_probability": float(saved_submission["target"].mean()),
                   "all_sanity_checks_passed": True},
}
(OUTPUT_DIR / "baseline_metrics.json").write_text(json.dumps(run_record, indent=2, allow_nan=False) + "\n")
print("All submission checks passed; raw hashes are unchanged.")
print("Aggregate run record: outputs/baseline_metrics.json")
print("Probability range:", run_record["submission"]["min_probability"], "to", run_record["submission"]["max_probability"])

All submission checks passed; raw hashes are unchanged.
Aggregate run record: outputs/baseline_metrics.json
Probability range: 0.0 to 1.0
